# 03 — Export Model → C Header

Convert trained model (bất kỳ loại nào) thành C if/else code để chạy trên ESP8266:
- **Decision Tree / Random Forest**: Export tree structure trực tiếp
- **Logistic Regression / Gaussian NB**: Train Decision Tree từ predictions để giữ accuracy
- **Output**: `edge_firmware/include/plant_classifier.h`
- **RAM footprint**: chỉ 3 float local variables khi inference
- **Flash**: vài KB C code


In [ ]:
import pickle
import numpy as np
from pathlib import Path
from sklearn.tree import _tree, DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline

MODELS_DIR   = Path('../models')
FIRMWARE_DIR = Path('../../edge_firmware/include')
FIRMWARE_DIR.mkdir(parents=True, exist_ok=True)

with open(MODELS_DIR / 'plant_classifier.pkl', 'rb') as f:
    meta = pickle.load(f)

original_model = meta['model']
model_name     = meta['model_name']
FEATURES       = meta['features']
CLASS_NAMES    = meta['class_names']
N_CLASSES      = meta['n_classes']
test_acc       = meta['test_accuracy']

print(f'Loaded: {model_name} | Test acc: {test_acc*100:.1f}%')
print(f'Features: {FEATURES}')
print(f'Classes: {CLASS_NAMES}')
print(f'Model type: {type(original_model).__name__}')


Model loaded | Test acc: 85.0%
Features: ['temperature', 'air_humidity', 'soil_moisture']
Classes: ['healthy', 'warning', 'critical']


AttributeError: 'RandomForestClassifier' object has no attribute 'get_depth'

In [ ]:
# ── Flexible C code generation based on model type ──────────────────────

def tree_to_c(clf, feature_names, class_names, tree_var_name='tree'):
    """Convert Decision Tree to C if/else code."""
    tree_ = clf.tree_
    feat_map = [
        feature_names[i] if i != _tree.TREE_UNDEFINED else '?'
        for i in tree_.feature
    ]

    lines = []

    def recurse(node, depth):
        pad = '    ' * depth

        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            feat = feat_map[node]
            thr = tree_.threshold[node]

            lines.append(f'{pad}if ({feat} <= {thr:.4f}f) {{')
            recurse(tree_.children_left[node], depth + 1)
            lines.append(f'{pad}}} else {{')
            recurse(tree_.children_right[node], depth + 1)
            lines.append(f'{pad}}}')
        else:
            cls_idx = int(np.argmax(tree_.value[node][0]))
            cls_name = class_names[cls_idx]
            lines.append(f'{pad}return {cls_idx};  // {cls_name}')

    recurse(0, 1)
    return '\n'.join(lines)


def rf_to_c(model, feature_names, class_names):
    """Convert Random Forest to C with voting from all trees."""
    n_trees = len(model.estimators_)
    n_classes = len(class_names)
    
    lines = [
        f'{" " * 4}// Random Forest ({n_trees} trees)',
        f'{" " * 4}int votes[{n_classes}] = {{0}};',
        '',
    ]
    
    for tree_idx, tree in enumerate(model.estimators_):
        lines.append(f'{" " * 4}// Tree {tree_idx}')
        tree_pred = tree_to_c(tree, feature_names, class_names, f'tree_{tree_idx}')
        # Indent tree code by 4 more spaces
        tree_pred_indented = '\n'.join(f'    {line}' for line in tree_pred.split('\n'))
        lines.append(f'{tree_pred_indented}')
        # Add voting - but need to capture tree return value
        # Better approach: inline each tree prediction
    
    # Better approach: for each sample, run each tree and vote
    lines = [
        f'{" " * 4}// Random Forest voting ({n_trees} trees)',
        f'{" " * 4}int votes[{n_classes}] = {{0}};',
        f'{" " * 4}int tree_pred;',
    ]
    
    for tree_idx, tree in enumerate(model.estimators_):
        tree_ = tree.tree_
        feat_map = [
            feature_names[i] if i != _tree.TREE_UNDEFINED else '?'
            for i in tree_.feature
        ]
        
        lines.append(f'')
        lines.append(f'{" " * 4}// Tree {tree_idx}')
        lines.append(f'{" " * 4}{{')
        
        def recurse_vote(node, depth):
            pad = '    ' * (depth + 1)

            if tree_.feature[node] != _tree.TREE_UNDEFINED:
                feat = feat_map[node]
                thr = tree_.threshold[node]

                lines.append(f'{pad}if ({feat} <= {thr:.4f}f) {{')
                recurse_vote(tree_.children_left[node], depth + 1)
                lines.append(f'{pad}}} else {{')
                recurse_vote(tree_.children_right[node], depth + 1)
                lines.append(f'{pad}}}')
            else:
                cls_idx = int(np.argmax(tree_.value[node][0]))
                cls_name = class_names[cls_idx]
                lines.append(f'{pad}tree_pred = {cls_idx};  // {cls_name}')

        recurse_vote(0, 1)
        lines.append(f'{" " * 4}}}')
        lines.append(f'{" " * 4}votes[tree_pred]++;')
    
    lines.extend([
        f'',
        f'{" " * 4}// Find class with most votes',
        f'{" " * 4}int max_votes = votes[0];',
        f'{" " * 4}int best_class = 0;',
        f'{" " * 4}for (int i = 1; i < {n_classes}; i++) {{',
        f'{" " * 8}if (votes[i] > max_votes) {{',
        f'{" " * 12}max_votes = votes[i];',
        f'{" " * 12}best_class = i;',
        f'{" " * 8}}}',
        f'{" " * 4}}}',
        f'{" " * 4}return best_class;',
    ])
    
    return '\n'.join(lines)


def logreg_to_c(model, feature_names, class_names):
    """Convert Logistic Regression to C sigmoid equation."""
    
    if isinstance(model, Pipeline):
        # Extract LogisticRegression from Pipeline
        lr = model.named_steps['logistic']
    else:
        lr = model
    
    n_classes = len(class_names)
    
    if n_classes == 2:
        # Binary classification
        coef = lr.coef_[0]
        intercept = lr.intercept_[0]
        
        lines = [
            f'{" " * 4}// Binary classification (Logistic Regression)',
            f'{" " * 4}float logit = {intercept:.6f}f;',
        ]
        for i, feat in enumerate(feature_names):
            lines.append(f'{" " * 4}logit += {coef[i]:.6f}f * {feat};')
        
        lines.extend([
            f'{" " * 4}float prob = 1.0f / (1.0f + expf(-logit));',
            f'{" " * 4}return (prob > 0.5f) ? 1 : 0;',
        ])
        return '\n'.join(lines)
    else:
        # Multiclass (One-vs-Rest)
        lines = [f'{" " * 4}// Multiclass Logistic Regression']
        lines.append(f'{" " * 4}float scores[{n_classes}];')
        
        for cls in range(n_classes):
            coef = lr.coef_[cls]
            intercept = lr.intercept_[cls]
            lines.append(f'{" " * 4}scores[{cls}] = {intercept:.6f}f;')
            for i, feat in enumerate(feature_names):
                lines.append(f'{" " * 4}scores[{cls}] += {coef[i]:.6f}f * {feat};')
        
        lines.append(f'{" " * 4}int max_class = 0;')
        lines.append(f'{" " * 4}float max_score = scores[0];')
        lines.append(f'{" " * 4}for (int i = 1; i < {n_classes}; i++) {{')
        lines.append(f'{" " * 8}if (scores[i] > max_score) {{')
        lines.append(f'{" " * 12}max_score = scores[i];')
        lines.append(f'{" " * 12}max_class = i;')
        lines.append(f'{" " * 8}}}')
        lines.append(f'{" " * 4}}}')
        lines.append(f'{" " * 4}return max_class;')
        
        return '\n'.join(lines)


def nb_to_c(model, feature_names, class_names):
    """Convert Gaussian Naive Bayes to C."""
    
    n_classes = len(class_names)
    lines = [f'{" " * 4}// Gaussian Naive Bayes']
    lines.append(f'{" " * 4}float log_probs[{n_classes}];')
    
    theta = model.theta_
    var = model.var_
    
    for cls in range(n_classes):
        prior = np.log(model.class_prior_[cls])
        lines.append(f'{" " * 4}log_probs[{cls}] = {prior:.6f}f;')
        
        for feat_idx, feat in enumerate(feature_names):
            mu = theta[cls][feat_idx]
            sigma2 = var[cls][feat_idx]
            coef = -0.5 / sigma2
            bias = -0.5 * np.log(2 * np.pi * sigma2)
            
            lines.append(f'{" " * 4}log_probs[{cls}] += {coef:.6f}f * ({feat} - {mu:.4f}f) * ({feat} - {mu:.4f}f) + {bias:.6f}f;')
    
    lines.append(f'{" " * 4}int max_class = 0;')
    lines.append(f'{" " * 4}float max_prob = log_probs[0];')
    lines.append(f'{" " * 4}for (int i = 1; i < {n_classes}; i++) {{')
    lines.append(f'{" " * 8}if (log_probs[i] > max_prob) {{')
    lines.append(f'{" " * 12}max_prob = log_probs[i];')
    lines.append(f'{" " * 12}max_class = i;')
    lines.append(f'{" " * 8}}}')
    lines.append(f'{" " * 4}}}')
    lines.append(f'{" " * 4}return max_class;')
    
    return '\n'.join(lines)


# ── Generate C code based on model type ──────────────────────────────────

if isinstance(original_model, DecisionTreeClassifier):
    print('→ Exporting Decision Tree directly')
    c_body = tree_to_c(original_model, FEATURES, CLASS_NAMES)
    model_type_desc = 'Decision Tree'

elif isinstance(original_model, RandomForestClassifier):
    print(f'→ Exporting Random Forest ({len(original_model.estimators_)} trees) with voting')
    c_body = rf_to_c(original_model, FEATURES, CLASS_NAMES)
    model_type_desc = f'Random Forest ({len(original_model.estimators_)} trees)'

elif isinstance(original_model, Pipeline):
    # Check what's inside the pipeline
    step_name, step = original_model.steps[-1]
    if isinstance(step, LogisticRegression):
        print('→ Exporting Logistic Regression (linear equation)')
        c_body = logreg_to_c(original_model, FEATURES, CLASS_NAMES)
        model_type_desc = 'Logistic Regression'
    else:
        raise ValueError(f'Unknown pipeline step: {step_name}')

elif isinstance(original_model, GaussianNB):
    print('→ Exporting Gaussian Naive Bayes')
    c_body = nb_to_c(original_model, FEATURES, CLASS_NAMES)
    model_type_desc = 'Gaussian Naive Bayes'

else:
    raise ValueError(f'Model type {type(original_model).__name__} not supported')

print(f'✓ C code generated ({len(c_body.splitlines())} lines)')


    if (soil_moisture <= 19.9609f) {
        return 2;  // critical
    } else {
        if (soil_moisture <= 30.0360f) {
            if (temperature <= 18.3061f) {
                return 1;  // warning
            } else {
                if (temperature <= 19.7795f) {
                    return 1;  // warning
                } else {
                    if (air_humidity <= 52.2397f) {
                        if (temperature <= 24.8501f) {
                            return 1;  // warning
                        } else {
                            return 1;  // warning
                      
...[truncated]


In [ ]:
# ── Verify C code correctness ────────────────────────────────────────────

if isinstance(original_model, DecisionTreeClassifier):
    
    def py_classify(temperature, air_humidity, soil_moisture):
        """Simulate C code traversal using tree."""
        tree_ = original_model.tree_
        vals = [temperature, air_humidity, soil_moisture]
        node = 0

        while tree_.feature[node] != _tree.TREE_UNDEFINED:
            idx = tree_.feature[node]
            if vals[idx] <= tree_.threshold[node]:
                node = tree_.children_left[node]
            else:
                node = tree_.children_right[node]

        return int(np.argmax(tree_.value[node][0]))

elif isinstance(original_model, RandomForestClassifier):
    
    def py_classify(temperature, air_humidity, soil_moisture):
        """Simulate C code voting from all trees."""
        n_trees = len(original_model.estimators_)
        n_classes = len(CLASS_NAMES)
        votes = [0] * n_classes
        
        for tree in original_model.estimators_:
            tree_ = tree.tree_
            vals = [temperature, air_humidity, soil_moisture]
            node = 0
            
            while tree_.feature[node] != _tree.TREE_UNDEFINED:
                idx = tree_.feature[node]
                if vals[idx] <= tree_.threshold[node]:
                    node = tree_.children_left[node]
                else:
                    node = tree_.children_right[node]
            
            tree_pred = int(np.argmax(tree_.value[node][0]))
            votes[tree_pred] += 1
        
        return votes.index(max(votes))

test_cases = [
    (25.0, 60.0, 45.0),
    (35.0, 20.0, 10.0),
    (20.0, 80.0, 70.0),
    (30.0, 40.0, 25.0)
]

print('Verification (C code will produce same results):')
for temp, rh, soil in test_cases:
    sk = int(original_model.predict([[temp, rh, soil]])[0])
    py = py_classify(temp, rh, soil)
    ok = '✓' if sk == py else '✗'
    print(f'{ok} {temp}°C {rh}% {soil}% → {CLASS_NAMES[sk]}')


Verification:
✓ 25.0°C 60.0% 45.0% → healthy
✓ 35.0°C 20.0% 10.0% → critical
✓ 20.0°C 80.0% 70.0% → healthy
✓ 30.0°C 40.0% 25.0% → warning


In [ ]:
# ── Generate C header ────────────────────────────────────────────────────

class_enum = '\n'.join([
    f'    HEALTH_{n.upper()} = {i},'
    for i, n in enumerate(CLASS_NAMES)
])

class_name_arr = ', '.join(f'"{n}"' for n in CLASS_NAMES)

include_math = '#include <math.h>' if 'Logistic' in model_type_desc else ''

header = f'''// AUTO-GENERATED — DO NOT EDIT
// Model: {model_name} → {model_type_desc}
// Test accuracy: {test_acc*100:.1f}%
#pragma once
#include <stdint.h>
{include_math}

#define N_HEALTH_CLASSES {N_CLASSES}

enum HealthStatus : uint8_t {{
{class_enum}
}};

static const char* const HEALTH_NAMES[] = {{ {class_name_arr} }};

inline HealthStatus classifyPlantHealth(
    float temperature,
    float air_humidity,
    float soil_moisture
) {{
{c_body}
}}
'''

out_path = FIRMWARE_DIR / 'plant_classifier.h'
out_path.write_text(header, encoding='utf-8')

print(f'✅ Written: {out_path}')
print(f'Lines: {len(header.splitlines())}')
print(f'Model exported: {model_name} ({model_type_desc})')


Written: ../../edge_firmware/include/plant_classifier.h
Lines: 79


In [ ]:
# ── RAM / Flash estimate ─────────────────────────────────────────────────

header_bytes = out_path.stat().st_size
runtime_ram = 12

print('=== ESP8266 Budget ===')
print(f'Flash: {header_bytes} bytes ({header_bytes/1024:.1f} KB)')
print(f'RAM:   {runtime_ram} bytes')
print()
print('Next: flash edge_firmware with generated header.')


=== ESP8266 Budget ===
Flash: 2497 bytes (2.4 KB)
RAM:   12 bytes

Next: flash edge_firmware with generated header.
